# Feature Engineering

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import ast

In [71]:
clean_df = pd.read_csv("../data/cleaned.csv")

In [72]:
clean_df["ingredient_names"] = clean_df["ingredient_names"].apply(ast.literal_eval)

In [73]:
features = [
    "manufacturer",
    "dosage_form",
    "pack_size",
    "pack_unit",
    "num_active_ingredients",
    "therapeutic_class",
    "primary_strength",
    "ingredient_names"
]

target = "price_inr"

In [74]:
X = clean_df[features]
y = np.log1p(clean_df[target])

In [75]:
X.head()

,manufacturer,dosage_form,pack_size,pack_unit,num_active_ingredients,therapeutic_class,primary_strength,ingredient_names
0,Glaxo SmithKline Pharmaceuticals Ltd,tablet,10.0,strip,2,antibiotic,500mg,"[amoxycillin, clavulanic acid]"
1,Alembic Pharmaceuticals Ltd,tablet,5.0,strip,1,antibiotic,500mg,[azithromycin]
2,Glenmark Pharmaceuticals Ltd,syrup,100.0,bottle,2,bronchodilator,30mg/5ml,"[ambroxol, levosalbutamol]"
3,Sanofi India Ltd,tablet,10.0,strip,1,antihistamine,120mg,[fexofenadine]
4,Sanofi India Ltd,tablet,15.0,strip,1,other,25mg,[pheniramine]


In [76]:
y.head()

0    5.413519
1    4.893052
2    4.779123
3    5.392764
4    2.481568
Name: price_inr, dtype: float64

In [77]:
X.dtypes

manufacturer                  str
dosage_form                   str
pack_size                 float64
pack_unit                     str
num_active_ingredients      int64
therapeutic_class             str
primary_strength              str
ingredient_names           object
dtype: object

In [78]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [79]:
comp_price = (
    pd.DataFrame({
        "composition": clean_df.loc[X_train.index, "cleaned_composition"],
        "price": y_train
    })
    .groupby("composition")["price"]
    .median()
)

X_train["composition_price"] = (
    clean_df.loc[X_train.index, "cleaned_composition"]
    .map(comp_price)
)

X_test["composition_price"] = (
    clean_df.loc[X_test.index, "cleaned_composition"]
    .map(comp_price)
    .fillna(y_train.median())
)

In [80]:
top_manufacturers = (
    X_train["manufacturer"]
    .value_counts()
    .nlargest(100)
    .index
)

X_train = X_train.copy()
X_test = X_test.copy()

X_train["manufacturer"] = X_train["manufacturer"].apply(
    lambda x: x if x in top_manufacturers else "Other"
)

X_test["manufacturer"] = X_test["manufacturer"].apply(
    lambda x: x if x in top_manufacturers else "Other"
)

In [81]:
print(X_train["manufacturer"].nunique())
print(X_test["manufacturer"].nunique())

101
101


In [82]:
categorical_features = [
    "manufacturer",
    "dosage_form",
    "pack_unit",
    "therapeutic_class"
]

numerical_features = [
    "pack_size",
    "num_active_ingredients",
    "composition_price",
]

ingredient_feature = "ingredient_names"

In [83]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_features])

X_train_encoded = encoder.transform(X_train[categorical_features])

X_test_encoded = encoder.transform(X_test[categorical_features])

In [84]:
encoded_train_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
encoded_test_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)
encoded_train_df.shape

(203171, 134)

In [85]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()

X_train_ingredients = mlb.fit_transform(
    X_train["ingredient_names"]
)

X_test_ingredients = mlb.transform(
    X_test["ingredient_names"]
)

c:\Users\habul\Desktop\Programs\Projects\Medicine-alternative-finder\venv\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['actarit', 'bacampicillin', 'benzthiazide', 'carbachol', 'certoparin', 'charcoal', 'dextran 70', 'diastase', 'enalaprilat', 'endoxifen', 'finerenone', 'garenoxacin', 'golimumab', 'ichthyol pale', 'idoxuridine', 'inactivated corona virus vaccine', 'inactivated hepatitis a vaccine', 'infliximab', 'inotuzumab ozogamacin', 'isavuconazole', 'lactobacillus brevis', 'lactobacillus salivarius', 'laropiprant', 'liposomal dithranol', 'lorlatinib', 'maraviroc', 'nattokinase', 'nystatin', 'pazufloxazin', 'pertuzumab', 'phenindione', 'polidocanol', 'potassium glycerophosphate', 'prednicarbate', 'propyl thiouracil', 'recombinant human parathyroid hormone', 'revaprazan', 'telbivudine', 'trabectedin', 'trithioparamethoxy phenylpropene', 'umeclidinium', 'varenicline', 'whole virion'] will be ignored
  warnings.warn(


In [86]:
ingredient_train_df = pd.DataFrame(
    X_train_ingredients,
    columns=mlb.classes_,
    index=X_train.index
)

ingredient_test_df = pd.DataFrame(
    X_test_ingredients,
    columns=mlb.classes_,
    index=X_test.index
)

In [87]:
def extract_strength(value):
    if pd.isna(value):
        return None

    value = str(value)

    match = re.search(r"\d+\.?\d*", value)

    if match:
        return float(match.group())

    return None

In [88]:
X_train_strength = X_train["primary_strength"].apply(extract_strength)

X_test_strength = X_test["primary_strength"].apply(extract_strength)

X_train[
    ["primary_strength"]
].head(10)

,primary_strength
161758,NaN
71960,5mg
33490,1000mg
157398,NaN
212150,5mg/5ml
144180,10mg
173476,200mg
115648,NaN
210983,200mg
142200,200mg


In [89]:
X_train_strength.head(10)

161758       NaN
71960        5.0
33490     1000.0
157398       NaN
212150       5.0
144180      10.0
173476     200.0
115648       NaN
210983     200.0
142200     200.0
Name: primary_strength, dtype: float64

In [90]:
X_train_final = pd.concat(
    [
        X_train[numerical_features],
        X_train_strength.rename("primary_strength_value"),
        encoded_train_df,
        ingredient_train_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test[numerical_features],
        X_test_strength.rename("primary_strength_value"),
        encoded_test_df,
        ingredient_test_df
    ],
    axis=1
)

X_train_final.shape

(203171, 1816)

In [91]:
X_train_final.head()

,pack_size,num_active_ingredients,composition_price,primary_strength_value,manufacturer_Aamorb Pharmaceuticals Pvt Ltd,manufacturer_Abbott,manufacturer_Ajanta Pharma Ltd,manufacturer_Akumentis Healthcare Ltd,manufacturer_Albia Biocare,manufacturer_Alembic Pharmaceuticals Ltd,...,zinc sulfate,zinc sulphate monohydrate,ziprasidone,zoledronic acid,zolmitriptan,zolpidem,zonisamide,zopiclone,zotepine,zuclopenthixol
161758,10.0,2,3.354106,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
71960,60.0,2,3.394756,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
33490,NaN,1,4.178533,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
157398,10.0,2,4.864453,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
212150,100.0,2,4.290459,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [92]:
X_train_final.isnull().sum().sort_values(ascending=False).head(20)

primary_strength_value                  20156
pack_size                               17892
vortioxetine                                0
manufacturer_Corona Remedies Pvt Ltd        0
zinc bisglycinate                           0
zinc acetate                                0
manufacturer_Albia Biocare                  0
zuclopenthixol                              0
num_active_ingredients                      0
manufacturer_Ajanta Pharma Ltd              0
zinc carnosine                              0
zinc chloride                               0
zinc gluconate                              0
zinc oxide                                  0
zinc pyrithione                             0
zinc sulfate                                0
zinc sulphate monohydrate                   0
ziprasidone                                 0
zoledronic acid                             0
zolmitriptan                                0
dtype: int64

In [93]:
X_train[["pack_size", "primary_strength"]].describe()

,pack_size
count,185279.000000
mean,19.067504
std,37.317671
min,1.000000
25%,10.000000
50%,10.000000
75%,10.000000
max,5000.000000


In [94]:
X_train["primary_strength"].value_counts().head(20)

primary_strength
100mg     17375
200mg     15978
500mg     15474
50mg      11571
10mg      11531
250mg     10009
5mg        8818
20mg       7730
40mg       7683
1000mg     6098
30mg       5783
2mg        4754
75mg       3802
400mg      3518
25mg       3102
1mg        3091
4mg        2815
150mg      2711
80mg       2438
300mg      2235
Name: count, dtype: int64

In [95]:
pack_size_median = X_train["pack_size"].median()

strength_median = X_train_strength.median()

print("Pack Size Median:", pack_size_median)
print("Strength Median:", strength_median)

Pack Size Median: 10.0
Strength Median: 60.0


- Replacing Missing values with their respective median

In [96]:
X_train_final["pack_size"] = X_train_final["pack_size"].fillna(pack_size_median)

X_train_final["primary_strength_value"] = (
    X_train_final["primary_strength_value"]
    .fillna(strength_median)
)

X_test_final["pack_size"] = X_test_final["pack_size"].fillna(pack_size_median)

X_test_final["primary_strength_value"] = (
    X_test_final["primary_strength_value"]
    .fillna(strength_median)
)

In [97]:
X_test_final.isnull().sum().sum()

np.int64(0)

## Model Training

A Random Forest Regressor is used as the initial baseline model.

Random Forest is an ensemble learning algorithm that combines predictions from multiple decision trees, making it robust to noise and capable of learning complex nonlinear relationships between medicine characteristics and price.

In [98]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

rf_model = RandomForestRegressor(
    n_estimators=20,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_final, y_train)

y_pred_log = rf_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))

r2 = r2_score(y_test_actual, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

MAE : 116.11
RMSE: 3238.79
R²  : 0.0400


In [99]:
print("Train R²:", rf_model.score(X_train_final, y_train))
print("Test R² :", rf_model.score(X_test_final, y_test))

Train R²: 0.9025943707111812
Test R² : 0.8146500681523203


In [100]:
print(encoded_train_df.shape)
print(len(mlb.classes_))

(203171, 134)
1678


In [101]:
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import time

In [102]:
lgb_model = LGBMRegressor(
    objective="regression_l1",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

In [103]:
start = time.time()

lgb_model.fit(X_train_final, y_train)

end = time.time()

print(f"Training Time: {end-start:.2f} seconds")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.093855 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2105
[LightGBM] [Info] Number of data points in the train set: 203171, number of used features: 841
[LightGBM] [Info] Start training from score 4.382027
Training Time: 22.93 seconds


In [104]:
y_pred_log = lgb_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

In [105]:
mae = mean_absolute_error(y_test_actual, y_pred)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))

r2 = r2_score(y_test_actual, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² : {r2:.4f}")

MAE : 111.17
RMSE: 3109.66
R² : 0.1150


In [106]:
print("Train R²:", lgb_model.score(X_train_final, y_train))
print("Test R² :", lgb_model.score(X_test_final, y_test))

Train R²: 0.883128518234696
Test R² : 0.820022660559805


In [107]:
y.describe()

count    253964.000000
mean          4.456674
std           1.017458
min           0.000000
25%           3.891820
50%           4.382027
75%           4.948760
max          12.985400
Name: price_inr, dtype: float64

In [108]:
y.quantile([0.90, 0.95, 0.99, 0.999])

0.900     5.541264
0.950     6.066700
0.990     7.972466
0.999    10.188481
Name: price_inr, dtype: float64

In [109]:
high_price = clean_df["price_inr"] > 3000

print("Medicines above ₹3000:", high_price.sum())
print("Percentage:", high_price.mean() * 100)

Medicines above ₹3000: 2416
Percentage: 0.9513159345419036


- LGBModel was better

In [110]:
import joblib

joblib.dump(
    lgb_model,
    "../models/medicine_price_model.pkl"
)

['../models/medicine_price_model.pkl']

In [111]:
joblib.dump(
    encoder,
    "../models/onehot_encoder.pkl"
)

['../models/onehot_encoder.pkl']

In [112]:
joblib.dump(
    mlb,
    "../models/multilabel_binarizer.pkl"
)

['../models/multilabel_binarizer.pkl']

In [113]:
joblib.dump(
    X_train_final.columns.tolist(),
    "../models/model_columns.pkl"
)

['../models/model_columns.pkl']

In [114]:
medians = {
    "pack_size": pack_size_median,
    "primary_strength_value": strength_median
}

joblib.dump(
    medians,
    "../models/medians.pkl"
)

['../models/medians.pkl']

In [115]:
joblib.dump(
    comp_price,
    "../models/composition_price.pkl"
)

['../models/composition_price.pkl']

In [116]:
joblib.dump(
    top_manufacturers,
    "../models/top_manufacturers.pkl"
)

['../models/top_manufacturers.pkl']

In [117]:
import os

print(os.listdir("../models"))

['composition_price.pkl', 'medians.pkl', 'medicine_price_model.pkl', 'model_columns.pkl', 'multilabel_binarizer.pkl', 'onehot_encoder.pkl', 'top_manufacturers.pkl']


In [118]:
joblib.dump(clean_df, "../data/cleaned_df.pkl")

['../data/cleaned_df.pkl']